In [1]:
#Version 1 of water data visualisation 
# CRS: ING EPSG: 29902 

import pandas as pd 
import geopandas as gpd 
from shapely.geometry import Point
import leafmap
import ipyleaflet
import json
import matplotlib.pyplot as plt 
from ipywidgets import IntSlider, VBox, Label
from time import sleep 


In [ ]:


### Section 1 ###

## Loading and transforming data ## 

#Convert water data and stations into pandas dataframes 

dodder20 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_020.csv')
dodder30 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_030.csv')
dodder40 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_040.csv')

#Merge Dataframes - Axis 0 = Rows, Axis 1 = Columns
water_data_raw =  pd.concat([dodder20, dodder30, dodder40], axis=0)


stations = pd.read_csv('/Users/emmet/Desktop/WaterData/stations.csv')
rivers = gpd.read_file('/Users/emmet/Desktop/WaterData/dublin_rivers.geojson')
rivers = rivers.to_crs(epsg=4326)

# Rename columns to allow for later join
water_data = water_data_raw.rename(columns={'MonitoringStationCode': 'StationID'}) 
stations = stations.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})

# Convert Date Column to Datetype
water_data["SampleDate"] = pd.to_datetime(water_data["SampleDate"], format="%d/%m/%Y")

#Create point objects from Lat/Long columns 
geometry = [Point(xy) for xy in zip(stations["longitude"], stations["latitude"])]
geo_stations = gpd.GeoDataFrame(stations, geometry=geometry, crs="EPSG:4326")


#Outer Join (Merge) of two datasets for georeferenced chemistry data 
full_data = geo_stations.merge(water_data, on="StationID")





In [ ]:
#Visualisation of Layers

rivers = gpd.read_file('/Users/emmet/Desktop/WaterData/dublin_rivers.geojson')
rivers = rivers.to_crs(epsg=4326)


m = leafmap.Map()
m.add_basemap("CartoDB.DarkMatterNoLabels")
m.add_gdf(rivers, layer_name = "Stream Network")
m.add_gdf(full_data, layer_name = "Chemistry Data", zoom_to_layer=True)

m


In [ ]:
### Section 2 ###

## Date and Chemical Control ##

#pH Total Median

ph_data = full_data[full_data["ParameterName"] == "pH"]

# Compute median pH per station
ph_median = ph_data.groupby("StationID").agg({
    "Result": "median", 
    "latitude": "first", 
    "longitude": "first"
}).reset_index()

m = leafmap.Map(center=[53.3, -6.3], zoom=12)
m.add_basemap("CartoDB.DarkMatterNoLabels")
m.add_gdf(rivers, layer_name = "Stream Network")
m.add_heatmap(ph_median, x="longitude", y="latitude", value="Result", radius=40, name="pH Median Heatmap", zoom_to_layer=True)
#m.add_heatmap(ph_median, layer_name = "Chemistry Data", zoom_to_layer=True)
m


In [ ]:

# Compute median pH per station per month
ph_median_monthly = ph_data.groupby([ph_data["SampleDate"].dt.to_period("M"), "StationID"]).agg({
    "Result": "median", 
    "latitude": "first", 
    "longitude": "first"
}).reset_index()

# Convert SampleDate to formatted string for better readability and sorting
ph_median_monthly["SampleDate"] = ph_median_monthly["SampleDate"].astype(str)
ph_median_monthly = ph_median_monthly.sort_values("SampleDate")

# Print median pH values for debugging
print(ph_median_monthly.groupby("SampleDate")["Result"].median())

# Create an interactive Leaflet map
m = leafmap.Map(center=[53.3, -6.3], zoom=12)
m.add_basemap("CartoDB.DarkMatterNoLabels")

# Add river layer
m.add_gdf(rivers, layer_name="Rivers")

# Create heatmap layer container
heatmap_layer = ipyleaflet.Heatmap()
m.add_layer(heatmap_layer)

# Set heatmap gradient and max value
heatmap_layer.max = 1  # Since scaled values are between 0 and 1
heatmap_layer.blur = 5
heatmap_layer.radius = 50
heatmap_layer.gradient = {
    0.0: 'red',     # Low pH (acidic)
    0.5: 'yellow',  # Neutral
    1.0: 'blue'     # High pH (alkaline)
}

# Create a time slider widget sorted chronologically
months = sorted(ph_median_monthly["SampleDate"].unique(), key=lambda x: pd.to_datetime(x, format='%Y-%m'))
slider = IntSlider(min=0, max=len(months) - 1, step=1, description='Month:')
label = Label(value=f"Showing data for: {months[0]}")

def update_heatmap(change):
    current_period = months[slider.value]
    filtered_data = ph_median_monthly[ph_median_monthly["SampleDate"] == current_period].copy()
    
    # Normalize pH values between 0 and 1 for better gradient visualization
    min_pH, max_pH = 6, 8
    filtered_data["scaled_pH"] = (filtered_data["Result"] - min_pH) / (max_pH - min_pH)
    filtered_data["scaled_pH"] = filtered_data["scaled_pH"].clip(0, 1)  # Ensure values stay within range
    
    heatmap_layer.locations = filtered_data[["latitude", "longitude", "scaled_pH"]].dropna().values.tolist()
    label.value = f"Showing data for: {current_period}"
    print(f"Displaying data for: {current_period}")

slider.observe(update_heatmap, names='value')

# Show the map, label, and slider
display(VBox([m, label, slider]))

# Initialize with first period
default_period = months[0]
default_data = ph_median_monthly[ph_median_monthly["SampleDate"] == default_period].copy()
default_data["scaled_pH"] = (default_data["Result"] - 6) / 2
default_data["scaled_pH"] = default_data["scaled_pH"].clip(0, 1)
heatmap_layer.locations = default_data[["latitude", "longitude", "scaled_pH"]].dropna().values.tolist()


In [2]:
import pandas as pd 
import geopandas as gpd 
from shapely.geometry import Point
import leafmap
from ipywidgets import IntSlider, VBox, Output
from IPython.display import display as ipy_display

# Load and prepare data 
dodder20 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_020.csv')
dodder30 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_030.csv')
dodder40 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_040.csv')
water_data_raw = pd.concat([dodder20, dodder30, dodder40], axis=0)

stations = pd.read_csv('/Users/emmet/Desktop/WaterData/stations.csv')
rivers = gpd.read_file('/Users/emmet/Desktop/WaterData/dublin_rivers.geojson')

water_data = water_data_raw.rename(columns={'MonitoringStationCode': 'StationID'}) 
stations = stations.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})

water_data["SampleDate"] = pd.to_datetime(water_data["SampleDate"], format="%d/%m/%Y")

geometry = [Point(xy) for xy in zip(stations["longitude"], stations["latitude"])]
geo_stations = gpd.GeoDataFrame(stations, geometry=geometry, crs="EPSG:4326")

full_data = geo_stations.merge(water_data, on="StationID")
ph_data = full_data[full_data["ParameterName"] == "pH"]

ph_median_monthly = ph_data.groupby([ph_data["SampleDate"].dt.to_period("M"), "StationID"]).agg({
    "Result": "median", 
    "latitude": "first", 
    "longitude": "first"
}).reset_index()

ph_median_monthly["SampleDate"] = ph_median_monthly["SampleDate"].astype(str)
months = sorted(ph_median_monthly["SampleDate"].unique())

# Output widget for displaying pH values
output = Output()

# Slider widget
slider = IntSlider(min=0, max=len(months) - 1, step=1, description='Month:')

def update_output(change):
    current_period = months[slider.value]
    filtered_data = ph_median_monthly[ph_median_monthly["SampleDate"] == current_period].copy()
    
    with output:
        output.clear_output(wait=True)
        print(f"pH Values for {current_period}:")
        print(filtered_data[["StationID", "Result"]].to_string(index=False))

slider.observe(update_output, names='value')

# Display the map separately
m = leafmap.Map(center=[53.3, -6.3], zoom=12)
m.add_gdf(rivers, layer_name="Rivers")
ipy_display(m)  # Ensure the map is shown first

# Now display the slider and output below
ipy_display(VBox([slider, output]))

# Initialize with first month
default_period = months[0]
default_data = ph_median_monthly[ph_median_monthly["SampleDate"] == default_period].copy()
with output:
    print(f"pH Values for {default_period}:")
    print(default_data[["StationID", "Result"]].to_string(index=False))


Map(center=[53.3, -6.3], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

In [8]:
import pandas as pd 
import geopandas as gpd 
from shapely.geometry import Point
import leafmap
import ipyleaflet
from ipywidgets import IntSlider, VBox, Output
from IPython.display import display as ipy_display

# Load and prepare data 
dodder20 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_020.csv')
dodder30 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_030.csv')
dodder40 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_040.csv')
dodder50 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_050.csv')
water_data_raw = pd.concat([dodder20, dodder30, dodder40, dodder50], axis=0)

stations = pd.read_csv('/Users/emmet/Desktop/WaterData/stations.csv')
rivers = gpd.read_file('/Users/emmet/Desktop/WaterData/dublin_rivers.geojson')

water_data = water_data_raw.rename(columns={'MonitoringStationCode': 'StationID'}) 
stations = stations.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})

water_data["SampleDate"] = pd.to_datetime(water_data["SampleDate"], format="%d/%m/%Y")

geometry = [Point(xy) for xy in zip(stations["longitude"], stations["latitude"])]
geo_stations = gpd.GeoDataFrame(stations, geometry=geometry, crs="EPSG:4326")

full_data = geo_stations.merge(water_data, on="StationID")
ph_data = full_data[full_data["ParameterName"] == "pH"]

ph_median_monthly = ph_data.groupby([ph_data["SampleDate"].dt.to_period("M"), "StationID"]).agg({
    "Result": "median", 
    "latitude": "first", 
    "longitude": "first",
    "MonitoringStationName": "first"
}).reset_index()

ph_median_monthly["SampleDate"] = ph_median_monthly["SampleDate"].astype(str)
months = sorted(ph_median_monthly["SampleDate"].unique())

# Determine global min and max pH for consistent scaling
min_pH, max_pH = 6.8, 8.2

# Create map 
m = leafmap.Map(center=[53.3, -6.3], zoom=12)
m.add_basemap("CartoDB.DarkMatter")
m.add_gdf(rivers, layer_name="Rivers", opacity=0.6)
heatmap_layer = ipyleaflet.Heatmap()
heatmap_layer.blur = 50
heatmap_layer.radius = 50
heatmap_layer.max_zoom = 1
heatmap_layer.min_opacity = 0.9
heatmap_layer.gradient = {
    0.5: 'blue',
    0.7: 'green',
    1.0: 'red'
}
m.add_layer(heatmap_layer)

# Output widget for displaying pH values
output = Output()

# Slider widget
slider = IntSlider(min=0, max=len(months) - 1, step=1, description='Month:')

def update_visuals(change):
    current_period = months[slider.value]
    filtered_data = ph_median_monthly[ph_median_monthly["SampleDate"] == current_period].copy()
    
    # Normalize pH values using fixed scale
    filtered_data["scaled_pH"] = (filtered_data["Result"] - min_pH) / (max_pH - min_pH)
    filtered_data["scaled_pH"] = filtered_data["scaled_pH"].clip(0, 1)
    
    # Update heatmap layer
    heatmap_layer.locations = filtered_data[["latitude", "longitude", "scaled_pH"]].dropna().values.tolist()
    
    # Update output values
    with output:
        output.clear_output(wait=True)
        print(f"pH Values for {current_period}:")
        print(filtered_data[["MonitoringStationName", "Result"]].to_string(index=False))

slider.observe(update_visuals, names='value')

# Display map, slider, and output
ipy_display(m)
ipy_display(VBox([slider, output]))

# Initialize with first month
default_period = months[0]
default_data = ph_median_monthly[ph_median_monthly["SampleDate"] == default_period].copy()
default_data["scaled_pH"] = (default_data["Result"] - min_pH) / (max_pH - min_pH)
default_data["scaled_pH"] = default_data["scaled_pH"].clip(0, 1)
heatmap_layer.locations = default_data[["latitude", "longitude", "scaled_pH"]].dropna().values.tolist()
with output:
    print(f"pH Values for {default_period}:")
    print(default_data[["MonitoringStationName", "Result"]].to_string(index=False))


Map(center=[53.3, -6.3], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

In [9]:
import pandas as pd 
import geopandas as gpd 
from shapely.geometry import Point
import leafmap
import ipyleaflet
from ipywidgets import IntSlider, VBox, Output, Button
from IPython.display import display as ipy_display
import time
import threading

# Load and prepare data 
dodder20 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_020.csv')
dodder30 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_030.csv')
dodder40 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_040.csv')
dodder50 = pd.read_csv('/Users/emmet/Desktop/WaterData/Dodder_050.csv')
water_data_raw = pd.concat([dodder20, dodder30, dodder40, dodder50], axis=0)

stations = pd.read_csv('/Users/emmet/Desktop/WaterData/stations.csv')
rivers = gpd.read_file('/Users/emmet/Desktop/WaterData/dublin_rivers.geojson')

water_data = water_data_raw.rename(columns={'MonitoringStationCode': 'StationID'}) 
stations = stations.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})

water_data["SampleDate"] = pd.to_datetime(water_data["SampleDate"], format="%d/%m/%Y")

geometry = [Point(xy) for xy in zip(stations["longitude"], stations["latitude"])]
geo_stations = gpd.GeoDataFrame(stations, geometry=geometry, crs="EPSG:4326")

full_data = geo_stations.merge(water_data, on="StationID")
ph_data = full_data[full_data["ParameterName"] == "pH"]

ph_median_monthly = ph_data.groupby([ph_data["SampleDate"].dt.to_period("M"), "StationID"]).agg({
    "Result": "median", 
    "latitude": "first", 
    "longitude": "first",
    "MonitoringStationName": "first"
}).reset_index()

ph_median_monthly["SampleDate"] = ph_median_monthly["SampleDate"].astype(str)
months = sorted(ph_median_monthly["SampleDate"].unique())

# Determine global min and max pH for consistent scaling
min_pH, max_pH = 6.8, 8.2

# Create map 
m = leafmap.Map(center=[53.3, -6.3], zoom=12)
m.add_basemap("CartoDB.DarkMatter")
m.add_gdf(rivers, layer_name="Rivers")
heatmap_layer = ipyleaflet.Heatmap()
heatmap_layer.blur = 50
heatmap_layer.radius = 50
heatmap_layer.max_zoom = 1
heatmap_layer.min_opacity = 0.7
heatmap_layer.gradient = {
    0.5: 'blue',
    0.7: 'green',
    1.0: 'red'
}
m.add_layer(heatmap_layer)

# Output widget for displaying pH values
output = Output()

# Slider widget
slider = IntSlider(min=0, max=len(months) - 1, step=1, description='Month:')

# ✅ Play button for auto-advancing slider
play_button = Button(description="▶ Play", button_style='primary')

# ✅ Flag to control animation
is_playing = False

def update_visuals(change=None):
    """ Updates heatmap based on selected month. """
    current_period = months[slider.value]
    filtered_data = ph_median_monthly[ph_median_monthly["SampleDate"] == current_period].copy()
    
    # Normalize pH values using fixed scale
    filtered_data["scaled_pH"] = (filtered_data["Result"] - min_pH) / (max_pH - min_pH)
    filtered_data["scaled_pH"] = filtered_data["scaled_pH"].clip(0, 1)
    
    # Update heatmap layer
    heatmap_layer.locations = filtered_data[["latitude", "longitude", "scaled_pH"]].dropna().values.tolist()
    
    # Update output values
    with output:
        output.clear_output(wait=True)
        print(f"pH Values for {current_period}:")
        print(filtered_data[["MonitoringStationName", "Result"]].to_string(index=False))

def play_animation():
    """ Auto-advance slider until stopped. """
    global is_playing
    is_playing = not is_playing  # Toggle state
    play_button.description = "⏸ Pause" if is_playing else "▶ Play"
    
    def animate():
        while is_playing and slider.value < len(months) - 1:
            slider.value += 1
            update_visuals()
            time.sleep(0.5)  # Adjust speed of animation

    if is_playing:
        threading.Thread(target=animate, daemon=True).start()

# Attach event listeners
slider.observe(update_visuals, names='value')
play_button.on_click(lambda _: play_animation())

# Display map, controls, and output
ipy_display(m)
ipy_display(VBox([slider, play_button, output]))

# Initialize with first month
update_visuals()


Map(center=[53.3, -6.3], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

pH Values for 2007-04:
 MonitoringStationName  Result
Footbridge, Beaver Row     8.3
pH Values for 2007-04:
 MonitoringStationName  Result
Footbridge, Beaver Row     8.3
pH Values for 2007-05:
 MonitoringStationName  Result
Footbridge, Beaver Row     8.2
pH Values for 2007-05:
 MonitoringStationName  Result
Footbridge, Beaver Row     8.2
pH Values for 2007-06:
 MonitoringStationName  Result
     U/s Piperstown St     7.4
           Old Bawn Br     7.8
      New Br, Firhouse     8.0
 Br on Springfield Ave     8.1
Footbridge, Beaver Row     8.2
pH Values for 2007-06:
 MonitoringStationName  Result
     U/s Piperstown St     7.4
           Old Bawn Br     7.8
      New Br, Firhouse     8.0
 Br on Springfield Ave     8.1
Footbridge, Beaver Row     8.2
pH Values for 2007-07:
 MonitoringStationName  Result
Footbridge, Beaver Row     8.2
pH Values for 2007-07:
 MonitoringStationName  Result
Footbridge, Beaver Row     8.2
pH Values for 2007-08:
 MonitoringStationName  Result
     U/s Piperstow